# Matrix Transformations: Interactive Visual Explorer

Welcome to the interactive companion notebook for **Phase 01 - Lesson 03: Matrix Transformations**.

In this notebook, we visualize linear transformations as geometric operations on space:
- **Rotations, Scaling, Shearing, and Reflection**
- **Composition of Transformations** (and why $A B \neq B A$)
- **Determinant** as the spatial area scaling factor
- **Eigenvalues & Eigenvectors** as invariant direction vectors ($A v = \lambda v$)
- **Eigendecomposition** ($A = V D V^{-1}$)


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 11


## 1. Basis Vectors & 2D Grid Transformation

A 2D linear transformation matrix $[[a, b], [c, d]]$ maps standard basis vector $e_1 = [1, 0]^T$ to $[a, c]^T$ and $e_2 = [0, 1]^T$ to $[b, d]^T$.


In [ ]:
def rotation_2d(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def scaling_2d(sx, sy):
    return np.array([[sx, 0.0], [0.0, sy]])

def shearing_2d(kx, ky):
    return np.array([[1.0, kx], [ky, 1.0]])

def reflection_y():
    return np.array([[-1.0, 0.0], [0.0, 1.0]])

def plot_transformation(matrix, title="Transformation"):
    x = np.linspace(-2, 2, 9)
    y = np.linspace(-2, 2, 9)
    grid_x, grid_y = np.meshgrid(x, y)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    for i in range(grid_x.shape[0]):
        ax1.plot(grid_x[i, :], grid_y[i, :], color='gray', alpha=0.4, linestyle='--')
        ax1.plot(grid_x[:, i], grid_y[:, i], color='gray', alpha=0.4, linestyle='--')
    ax1.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1, color='blue', label='e1 (1,0)')
    ax1.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1, color='red', label='e2 (0,1)')
    ax1.set_xlim(-4, 4)
    ax1.set_ylim(-4, 4)
    ax1.set_aspect('equal')
    ax1.set_title("Original Grid & Basis")
    ax1.legend()
    
    pts = np.vstack([grid_x.flatten(), grid_y.flatten()])
    trans_pts = matrix @ pts
    trans_x = trans_pts[0, :].reshape(grid_x.shape)
    trans_y = trans_pts[1, :].reshape(grid_y.shape)
    
    for i in range(trans_x.shape[0]):
        ax2.plot(trans_x[i, :], trans_y[i, :], color='teal', alpha=0.5)
        ax2.plot(trans_x[:, i], trans_y[:, i], color='teal', alpha=0.5)
        
    te1 = matrix @ np.array([1, 0])
    te2 = matrix @ np.array([0, 1])
    ax2.quiver(0, 0, te1[0], te1[1], angles='xy', scale_units='xy', scale=1, color='blue', label='Transformed e1')
    ax2.quiver(0, 0, te2[0], te2[1], angles='xy', scale_units='xy', scale=1, color='red', label='Transformed e2')
    ax2.set_xlim(-4, 4)
    ax2.set_ylim(-4, 4)
    ax2.set_aspect('equal')
    ax2.set_title(title)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

plot_transformation(rotation_2d(np.pi / 4), "Rotation by 45 degrees")


## 2. Matrix Composition & Non-Commutativity

Applying transformation $A$ followed by $B$ is represented by the matrix product $B \cdot A$.
Because matrix multiplication is non-commutative ($B \cdot A \neq A \cdot B$), order matters.


In [ ]:
R = rotation_2d(np.pi / 2)
S = scaling_2d(2.0, 0.5)

RS = R @ S
SR = S @ R

point = np.array([1.0, 1.0])
p_rs = RS @ point
p_sr = SR @ point

print(f"Original Point: {point}")
print(f"Scale then Rotate (R @ S @ p): {p_rs}")
print(f"Rotate then Scale (S @ R @ p): {p_sr}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.quiver(0, 0, point[0], point[1], angles='xy', scale_units='xy', scale=1, color='gray', label='Original (1,1)')
ax.quiver(0, 0, p_rs[0], p_rs[1], angles='xy', scale_units='xy', scale=1, color='purple', label='R @ S (Scale then Rotate)')
ax.quiver(0, 0, p_sr[0], p_sr[1], angles='xy', scale_units='xy', scale=1, color='orange', label='S @ R (Rotate then Scale)')
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.set_title("Matrix Composition: R @ S vs S @ R")
ax.legend()
plt.show()


## 3. Determinant as Spatial Area Scaling

The determinant $\det(M)$ measures how much a matrix scales 2D area:
- $\det(M) = 1$: Area preserved (e.g. rotation, shearing)
- $\det(M) = 0$: Collapse to lower dimension (singular matrix)
- $\det(M) < 0$: Area scaled and orientation flipped (reflection)


In [ ]:
unit_square = np.array([[0, 1, 1, 0, 0], [0, 0, 1, 1, 0]])

matrices = [
    ("Rotation (45 deg)", rotation_2d(np.pi/4)),
    ("Scaling (2, 3)", scaling_2d(2, 3)),
    ("Shearing (kx=1)", shearing_2d(1, 0)),
    ("Reflection (Y-axis)", reflection_y()),
    ("Singular (Collapse)", np.array([[1, 2], [2, 4]]))
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (name, M) in zip(axes, matrices):
    trans = M @ unit_square
    det_val = np.linalg.det(M)
    ax.plot(unit_square[0, :], unit_square[1, :], 'k--', alpha=0.5, label='Unit Square (Area=1)')
    ax.fill(trans[0, :], trans[1, :], color='cyan', alpha=0.4, label='Transformed')
    ax.plot(trans[0, :], trans[1, :], color='blue')
    ax.set_xlim(-2.5, 3.5)
    ax.set_ylim(-2.5, 3.5)
    ax.set_aspect('equal')
    ax.set_title(f"{name}\ndet = {det_val:.2f}")

plt.tight_layout()
plt.show()


## 4. Eigenvalues & Eigenvectors Geometry

An eigenvector $v$ satisfies $A v = \lambda v$. Under transformation $A$, $v$ does not change its line of span; it is scaled by factor $\lambda$.


In [ ]:
A = np.array([[2.0, 1.0], [1.0, 2.0]])
eigenvalues, eigenvectors = np.linalg.eig(A)

print(f"Matrix A:\n{A}")
print(f"Eigenvalues: {eigenvalues}")
print(f"Eigenvectors:\n{eigenvectors}")

fig, ax = plt.subplots(figsize=(7, 7))

angles = np.linspace(0, 2 * np.pi, 24, endpoint=False)
for ang in angles:
    v = np.array([np.cos(ang), np.sin(ang)])
    Av = A @ v
    ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color='lightgray', alpha=0.7)
    ax.quiver(0, 0, Av[0], Av[1], angles='xy', scale_units='xy', scale=1, color='pink', alpha=0.3)

colors = ['crimson', 'darkgreen']
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    Av = A @ v
    lam = eigenvalues[i]
    ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color=colors[i], width=0.015, label=f'Eigenvector v{i+1} {v.round(2)}')
    ax.quiver(0, 0, Av[0], Av[1], angles='xy', scale_units='xy', scale=1, color=colors[i], alpha=0.5, linestyle=':', width=0.01, label=f'A @ v{i+1} (lambda={lam:.1f})')

ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.set_title("Eigenvectors Maintain Line of Span (Av = lambda * v)")
ax.legend()
plt.show()


## 5. Eigendecomposition: $A = V D V^{-1}$

Eigendecomposition expresses space transformation as three simple geometric operations:
1. **$V^{-1}$**: Rotate/change basis into eigenvector directions
2. **$D$**: Scale independently along each eigenvector axis by $\lambda_i$
3. **$V$**: Rotate back to standard basis


In [ ]:
B = np.array([[3.0, 1.0], [0.0, 2.0]])
vals, vecs = np.linalg.eig(B)
D = np.diag(vals)
V = vecs
V_inv = np.linalg.inv(V)

reconstructed = V @ D @ V_inv
print("Original Matrix B:\n", B)
print("Reconstructed V @ D @ V^-1:\n", reconstructed)
print("Exact Match?", np.allclose(B, reconstructed))


## 6. Lesson Exercises Playground

Interactive verification of Exercises 1, 2, and 3 from `docs/en.md`.


In [ ]:
print("--- Exercise 1: Unit Square Corner Transformation ---")
corners = np.array([[0, 1, 1, 0], [0, 0, 1, 1]])
R30 = rotation_2d(np.radians(30))
S15 = scaling_2d(1.5, 0.8)
Sh03 = shearing_2d(0.3, 0)

trans_corners = Sh03 @ S15 @ R30 @ corners
print("Transformed corners:\n", trans_corners.round(4))

print("\n--- Exercise 2: Characteristic Equation Verification ---")
M_ex2 = np.array([[4.0, 2.0], [1.0, 3.0]])
tr = np.trace(M_ex2)
det = np.linalg.det(M_ex2)
disc = tr**2 - 4*det
l1 = (tr + np.sqrt(disc)) / 2
l2 = (tr - np.sqrt(disc)) / 2
vals_np = np.linalg.eigvals(M_ex2)
print(f"Hand-computed eigenvalues: {l1:.1f}, {l2:.1f}")
print(f"NumPy eigenvalues:         {vals_np[0]:.1f}, {vals_np[1]:.1f}")

print("\n--- Exercise 3: Composition Determinants Product ---")
C_composed = Sh03 @ S15 @ R30
det_composed = np.linalg.det(C_composed)
det_product = np.linalg.det(Sh03) * np.linalg.det(S15) * np.linalg.det(R30)
print(f"det(Composed):  {det_composed:.4f}")
print(f"det(Product):   {det_product:.4f}")
print("Equal?", np.isclose(det_composed, det_product))
